In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/diabetic_data.csv")
print(df.shape)
print(df.columns.tolist())

(101766, 50)
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


In [2]:
print(df["readmitted"].value_counts())
miss = (df == "?").sum()
print(miss[miss > 0])

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
dtype: int64


In [3]:
import sys; sys.path.append("..")
from src.data_loader import load_and_clean
X, y, meta = load_and_clean()
print(X.shape, round(y.mean(), 4))
print(X.dtypes.value_counts())
print(X.isna().sum().sum())     # should be 0

(99340, 43) 0.1139
str      34
int64     9
Name: count, dtype: int64
0


In [4]:
import sys; sys.path.append("..")
import numpy as np
from src.data_loader import prepare_pool_and_test

info = prepare_pool_and_test()
print(info["input_dim"], info["n_pool"], info["n_test"])

pool = np.load("../data/processed/pool.npz")
test = np.load("../data/processed/global_test.npz")
print(pool["X"].shape, test["X"].shape)
print(pool["y"].mean().round(4), test["y"].mean().round(4))   # both near 0.114
print(np.isnan(pool["X"]).sum(), np.isnan(test["X"]).sum())   # both 0
print(len(set(pool["patient_nbr"]) & set(test["patient_nbr"])))   # must be 0

182 79567 19773
(79567, 182) (19773, 182)
0.114 0.1135
0 0
0


In [1]:
import sys; sys.path.append("..")
from src.partition import build_hospitals
from src.data_loader import load_hospital_data

summary = build_hospitals()
print(summary.to_string(index=False))
print("total:", summary["n_total"].sum())          # must equal 79567

Xtr, ytr, Xva, yva = load_hospital_data(0)
print(Xtr.shape, Xva.shape, ytr.mean().round(4))

 hospital  n_total  n_train  n_val  positive_rate  pct_age_<50  pct_age_50-69  pct_age_70+
        0    13586    12320   1266         0.2236        0.057          0.546        0.398
        1    23965    21571   2394         0.0682        0.147          0.316        0.537
        2    10273     9162   1111         0.2480        0.149          0.453        0.398
        3     9026     8173    853         0.0841        0.130          0.757        0.113
        4    22717    20496   2221         0.0480        0.250          0.212        0.538
total: 79567
(12320, 182) (1266, 182) 0.2243
